In [ ]:
import os

os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from umap import UMAP
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from tqdm import tqdm
import re

## Creating TF-IDF Vectors

In [ ]:
unfalltext_table_path = "data/D_Unfalltext_image_50000_subsample.csv"
unfalltext_table = pd.read_csv(unfalltext_table_path)
unfalltext_table.head(3)

In [ ]:
corpus = unfalltext_table['Text'].values
corpus

In [ ]:
# Remove numbers and special characters from the corpus
cleaned_corpus = [re.sub(r'[^a-zA-ZäöüÄÖÜß\s]', '', text) for text in corpus]
cleaned_corpus

In [ ]:
#nltk.download('stopwords')
german_stopwords = list(set(stopwords.words('german')))
print("Numb of german stopwords: ", len(german_stopwords))
german_stopwords[:5]

In [ ]:
# Step 1: TF-IDF Vectorization
# Remove german stopwords and include n-grams
vectorizer = TfidfVectorizer(
    stop_words=german_stopwords,  # Remove German stopwords
    ngram_range=(1,2)           # Use unigrams, bigrams
)
tfidf_matrix = vectorizer.fit_transform(cleaned_corpus)

In [ ]:
# 775K dimensional vector for each text
tfidf_matrix.shape

In [ ]:
vectorizer.get_feature_names_out()

## Defining the # of Clusters

In [ ]:
print("TF-IDF Matrix Shape: ", tfidf_matrix.shape)

# Use UMAP to reduce the dimensionality of the tfidf matrix
# Dim 100
umap_100 = UMAP(n_components=100, random_state=0)
umap_100_matrix = umap_100.fit_transform(tfidf_matrix)
print("UMAP 100 Matrix Shape: ", umap_100_matrix.shape)

# Dim 10
umap_10 = UMAP(n_components=10, random_state=0)
umap_10_matrix = umap_10.fit_transform(tfidf_matrix)
print("UMAP 100 Matrix Shape: ", umap_10_matrix.shape)

In [ ]:
def get_silhouette_scores(k_min, k_max, matrix):
    silhouette_scores = []
    k_values = range(k_min, k_max)  # Range of clusters to evaluate

    for k in tqdm(k_values):
        kmeans = KMeans(n_clusters=k, random_state=0)
        labels = kmeans.fit_predict(matrix)
        score = silhouette_score(matrix, labels)
        silhouette_scores.append(score)
    
    return silhouette_scores

In [ ]:
silhouette_scores_full_dim = get_silhouette_scores(k_min=5, k_max=15, matrix=tfidf_matrix)

In [ ]:
silhouette_scores_100_dim = get_silhouette_scores(k_min=5, k_max=15, matrix=umap_100_matrix)

In [ ]:
silhouette_scores_10_dim = get_silhouette_scores(k_min=5, k_max=15, matrix=umap_10_matrix)

In [ ]:
# Plot 3 silhouette scores
plt.figure(figsize=(12, 6))
plt.plot(range(5, 15), silhouette_scores_full_dim, label='Full TF-IDF', marker='o')
plt.plot(range(5, 15), silhouette_scores_100_dim, label='UMAP 100', marker='o')
plt.plot(range(5, 15), silhouette_scores_10_dim, label='UMAP 10', marker='o')
plt.ylim(-1,1)
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.legend()
plt.show()

In [ ]:
silhouette_scores_10_dim

## Exploring the Clusters

In [ ]:
numb_clusters = 11

In [ ]:
kmeans = KMeans(n_clusters=numb_clusters, random_state=0)
kmeans.fit(umap_100_matrix)

In [ ]:
kmeans.labels_

In [ ]:
# Step 3: Analyze Cluster Assignments
print("Cluster assignments:", kmeans.labels_)

# Print top terms in each cluster
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(numb_clusters):
    print(f"\nCluster {i}:")
    for ind in order_centroids[i, :10]:  # Top 10 terms per cluster
        print(f"  {terms[ind]}")

## Analysis with Human Labels

In [ ]:
from explorative.conventions import unfalltyp_mapping_eng

In [ ]:
unfall_table_path = "data/D_Unfall_RData.csv"
unfall_table = pd.read_csv(unfall_table_path)

In [ ]:
unfall_text_table_cluster_merged = unfall_table.merge(unfalltext_table, on='UN_KEY', how='left')

# Remove the rows not existing in unfalltext_table
unfall_text_table_cluster_merged = unfall_text_table_cluster_merged[unfall_text_table_cluster_merged.UN_KEY.isin(unfalltext_table.UN_KEY)]

# Add the cluster assignments to the merged table
unfall_text_table_cluster_merged['tfidf_cluster'] = kmeans.labels_
unfall_text_table_cluster_merged.shape

In [ ]:
unfall_text_table_cluster_merged.head(3)

In [ ]:
tfidf_cluster_per_human_label = unfall_text_table_cluster_merged.groupby('Unfalltyp')['tfidf_cluster'].value_counts().unstack().fillna(0)
tfidf_cluster_per_human_label

In [ ]:
# Show it as percentage accross the human label category
tfidf_cluster_per_human_label = unfall_text_table_cluster_merged.groupby('Unfalltyp')['tfidf_cluster'].value_counts().unstack().fillna(0)
tfidf_cluster_per_human_label_percentage = (tfidf_cluster_per_human_label.div(tfidf_cluster_per_human_label.sum(axis=1), axis=0) * 100).round(0)
tfidf_cluster_per_human_label_percentage

In [ ]:
# Plot for the distribution of LLM labels for each human label category
distr_plot_data = tfidf_cluster_per_human_label_percentage.copy()
distr_plot_data.index = distr_plot_data.index.map(lambda x: f"{int(x)}: {unfalltyp_mapping_eng.get(x, 'Unknown')}")
categories = distr_plot_data.index
tfidf_cluster = distr_plot_data.columns

# Set up the plot
fig, ax = plt.subplots(figsize=(16, 16))
ax.set_xlim(-0.5, len(categories)-0.5)  # Reversed xlim and ylim
ax.set_ylim(-0.5, len(tfidf_cluster)-0.5)

# Create the bubble chart
for i, category in enumerate(categories):
    for j, label in enumerate(tfidf_cluster):
        percentage = distr_plot_data.loc[category, label]
        if percentage > 0:  # Only plot non-zero percentages
            circle_size = percentage * 100  # Adjust size scaling factor
            ax.scatter(i, j, s=circle_size, color='skyblue', alpha=0.7, edgecolor='black')  # Swapped i and j
            circle_font = 'bold' if percentage > 20 else 'normal'
            circle_font_size = 20 if percentage > 20 else 10
            ax.text(i, j, f"{int(percentage)}%", ha='center', va='center', 
                    fontsize=circle_font_size, weight=circle_font)  # Swapped i and j

# Adjust ticks and labels
ax.set_xticks(range(len(categories)))  # Swapped range to categories
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=20)
ax.set_xlabel("Human Labels", fontsize=20)  # Swapped x-label
ax.set_yticks(range(len(tfidf_cluster)))  # Swapped range to llm_labels
ax.set_yticklabels(tfidf_cluster, fontsize=20)
ax.set_ylabel("TF-IDF Clusters", fontsize=20)  # Swapped y-label

# Add grid for better readability
ax.grid(True, linestyle='--', alpha=0.6)

# Add title
plt.title("TF-IDF Clusters vs Human Labels", fontsize=20)
plt.tight_layout()
plt.show()